# Customer Churn Analysis & Modeling

This notebook reproduces the complete churn modeling workflow used by the production app.

In [ ]:
import warnings
from pathlib import Path

import joblib
import pandas as pd
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parents[0]
DATA_PATH = PROJECT_ROOT / 'data' / 'Telco-Customer-Churn.csv'
MODEL_PATH = PROJECT_ROOT / 'models' / 'churn_model.pkl'

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna().reset_index(drop=True)

if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
X = pd.get_dummies(df.drop(columns=['Churn']), drop_first=False)
y = df['Churn']

numeric_cols = [c for c in ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen'] if c in X.columns]
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
}

scores = {}
trained = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    scores[name] = {
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred),
        'recall': recall_score(y_test, pred),
        'f1': f1_score(y_test, pred),
        'roc_auc': roc_auc_score(y_test, prob),
    }

scores_df = pd.DataFrame(scores).T.sort_values('roc_auc', ascending=False)
scores_df

In [ ]:
best_model_name = scores_df.index[0]
best_model = trained[best_model_name]

artifact = {
    'model': best_model,
    'model_name': best_model_name,
    'metrics': scores,
    'feature_columns': X.columns.tolist(),
    'numeric_columns': numeric_cols,
    'scaler': scaler,
}

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, MODEL_PATH)
print(f'Saved {best_model_name} to {MODEL_PATH}')

In [ ]:
plot_df = pd.DataFrame({
    'Model': scores_df.index,
    'ROC-AUC': scores_df['roc_auc'].values
})
fig = px.bar(plot_df, x='Model', y='ROC-AUC', color='Model', template='plotly_dark', title='Model Comparison')
fig.show()